# 12.3 自訂排序鍵值基礎：key 參數與具名函數（Named Functions）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_12-3_custom_sort_key_and_named_functions.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 Chapter 11 自訂函數定義（`def`）、回傳值（`return`）與 Chapter 12 前兩節之排序基礎。

---

### 學習導覽：照出隱藏特徵的 X 光機——客製化排序的啟蒙之門

在先前的學習中，我們排序的都是最單純的一維數值或字母（例如整數比數值大小、字串比 ASCII 字典序）。
然而在現實世界與 APCS 實作考題中，我們面對的資料往往複雜得多：
- 題目給定一群單字，要求**「依照單字的字母長度由短到長排列」**，而不是看首字母的字典序。
- 題目給定正負交雜的溫差讀數，要求**「依照離 0 度的絕對距離（絕對值）由小到大排列」**。
- 題目給定多筆二維學生資料 `(姓名, 成績)`，要求**「依照成績（第二個欄位）來決定排名」**。

如果只能比物件本身，我們該怎麼辦？難道要把資料拆散、排完後再想辦法拼回去嗎？
絕對不需要！Python 排序系統擁有一項享譽全球的強大設計——**`key=` 參數**。
本單元專門為零基礎初學者打造了一座平緩的「認知階梯」：在直接跳入晦澀的匿名語法之前，我們先利用第十一章學過的標準**具名函數（`def`）**與內建工具，徹底通透 `key=` 參數的「特徵投射（Projection）」本質！

在本單元中，我們將透過 6 個平緩微階梯，逐步建立客製化排序的核心思維：
1. **12.3.1 什麼是 `key=` 參數？**「投射轉換（Projection）」比喻：依特徵值比大小。
2. **12.3.2 內建函數作為鍵值**：`key=abs`（依絕對值排序）、`key=len`（依長度排序）。
3. **12.3.3 自訂具名函數作為鍵值**：使用 `def get_feature(x):` 傳入 `key=get_feature`。
4. **12.3.4 元組第二欄位排序**：`def get_second(item): return item[1]`（成績、年齡排序）。
5. **12.3.5 字典走訪排序**：以 `sorted(d.items(), key=...)` 依字典的 Value 進行排序。
6. **12.3.6 排序除錯心法**：如何單獨測試 key 函數確認特徵值抽取無誤。

讓我們戴上特徵透視眼鏡，解鎖千變萬化的自訂排序魔法！

### 12.3.1 什麼是 `key=` 參數？「投射轉換（Projection）」比喻：依特徵值比大小

#### 1. 生活故事比喻：行李箱安檢的 X 光透視機
想像你走進國際機場的海關行李檢查站，輸送帶上有各式各樣造型各異的行李箱：有的是紅色硬殼箱、有的是黑色帆布袋、有的是名牌皮箱。
如果海關官員想將這些行李箱「由輕到重排好」，官員該怎麼做？
官員不需要把行李箱的皮革拆開。他只需要讓每個行李箱依序通過一台「電子地磅機（秤重機）」。
- 箱子 A 放上去，地磅機螢幕顯示：`12 kg`。
- 箱子 B 放上去，地磅機螢幕顯示：`5 kg`。
- 箱子 C 放上去，地磅機螢幕顯示：`20 kg`。
海關官員在心裡排好的是這些數字：`5 < 12 < 20`，最後按照這個順序將「原本完整的行李箱」依序排在地上：箱子 B、箱子 A、箱子 C。
這台只負責抽取特徵數字的電子秤，在 Python 排序中就是 **`key` 函數**！

#### 2. 底層運作機制：特徵投射（Key Projection）兩步曲
當我們向 `sort()` 或 `sorted()` 傳入 `key=函數名牌` 時，排序引擎在內部進行了極為優雅的兩步處理：
1. **一對一特徵抽取（Projection）**：  
   引擎拿著你提供的函數，對待排序串列中的每一個元素呼叫一次：$K_i = Key(Item_i)$，得到一把把看不見的「特徵鑰匙（Key）」。
2. **依鑰匙比大小，維持原物不動**：  
   在比較誰大誰小時，演算法不是拿原物件去比，而是比對各自算出來的 $K_i$！比較出先後名次後，直接將原本完整無損的元素擺到正確位置上。
排序完成後，原本的物件型態、內部結構毫髮無傷，只是位置被巧妙重新佈局了！

#### 3. 初學者常見陷阱：誤以為原物件會被 key 函數覆蓋
許多初學者常常擔心地問：「如果我指定依照字串長度排序，排序完後我的串列會不會變成一堆數字長度 `[3, 5, 2]`？」
答案是：**絕對不會！** `key` 函數只是一面「臨時鏡子」，排序工人只是看著鏡子裡的倒影來決定排隊順序，排完之後鏡子拿走，站在隊列裡的依然是原本活生生的原物件。

#### 4. APCS 實戰視野
理解「特徵值投射」是征服所有進階競賽題的關鍵分水嶺。無論資料結構是長條字串、座標點、學生資料結構，只要能寫出一個提取特徵值的函數，世間萬物皆可隨心所欲地精準排序。

In [ ]:
# 範例 12.3.1：感知 key 參數的投射比大小模型
# 宣告一組包含不同字母長度的單字
words = ["elephant", "ox", "cat", "hippopotamus", "dog"]
print("原始單字清單：", words)

# 定義一個特徵抽取函數：傳入單字，回傳其字母長度
def get_length(word):
    return len(word)

# 觀察特徵抽取的過程：
for w in words:
    print(f"單字: {w:<12} -> 特徵長度: {get_length(w)}")

# 將 get_length 函數名牌交給 key 參數進行排序
by_length = sorted(words, key=get_length)

print("\n依長度由短到長排序後的結果：", by_length)
# 驗證：清單中的元素依然是英文字串，完全沒有被數字長度替換！

In [ ]:
# 填空題 12.3.1：依字串長度自訂排序
# 任務：將多句口號依照字串總長度由短到長排序。
slogans = ["Go!", "Keep learning python", "Never give up", "Hi"]

def word_len(s):
    # 請回傳字串 s 的長度
    return ___(s)

# 請將 word_len 作為 key 參數傳入 sorted 函數
sorted_slogans = sorted(slogans, key=___)

print("依照長度排序的口號：", sorted_slogans)

In [ ]:
# ==========================================
# [4] Code 練習題 12.3.1
# 任務說明：
# 給定一組字串清單 phrases。
# 請定義一個特徵函數 text_length(text)，
# 使用 sorted(..., key=text_length) 將字串由短到長排序，
# 並印出排序結果，以及排在最長（最後一個）的字串與其長度。
#
# 【公開測試資料 1】
# phrases = ["apple", "pie", "banana", "watermelon", "fig"]
# 預期輸出：
# 依長度排序： ['pie', 'fig', 'apple', 'banana', 'watermelon']
# 最長字串： watermelon (長度 10)
#
# 【公開測試資料 2】
# phrases = ["code", "c", "python"]
# 預期輸出：
# 依長度排序： ['c', 'code', 'python']
# 最長字串： python (長度 6)
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def text_length(text):
    return len(text)

phrases = ["apple", "pie", "banana", "watermelon", "fig"]
res = sorted(phrases, key=text_length)
print("依長度排序：", res)
print(f"最長字串： {res[-1]} (長度 {len(res[-1])})")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.3.1
# 任務說明：
# 某網站有一組文章標題清單：
# titles = ["How to Code", "Python", "AI Revolution in Modern World", "APCS"]
# 請寫出完整程式，定義一個取得字串長度的函數，
# 將文章標題依照「標題字串長度由長到短（降序）」排序並印出。
# （提示：可結合 key 與 reverse 參數）
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def get_len(text):
    return len(text)

titles = ["How to Code", "Python", "AI Revolution in Modern World", "APCS"]
long_to_short = sorted(titles, key=get_len, reverse=True)
print("標題依長度降序排列：")
for t in long_to_short:
    print(f"  長度 {len(t):2d}: {t}")

### 12.3.2 內建函數作為鍵值：`key=abs`（依絕對值排序）、`key=len`（依長度排序）

#### 1. 生活故事比喻：現成的工廠量尺
當我們想要量測木板的長度時，我們不需要親自去砍樹、鋸木頭造一把全新尺子。工具箱裡早就躺著一把現成、精準的捲尺（如 `len`）和一把水平儀（如 `abs`），直接拿出來用即可！
在 Python 中，許多內建函數天生就是完美的「特徵抽取器」：
- **`len`**：傳入任何字串或容器，瞬間量測出它的長度。
- **`abs`**：傳入任何正負整數或浮點數，瞬間計算出它離原點 0 的絕對值距離。
當我們要依長度或絕對值排序時，根本不需要花時間用 `def` 自己造輪子，直接將 **`key=len`** 或 **`key=abs`** 填上去，一行程式碼就能創造奇蹟！

#### 2. 底層運作機制：函數名牌即是第一類物件（First-class Object）
在 Python 哲學中，「函數是一等公民」。這意味著函數名（如 `len`、`abs`）就像整數 `5` 或字串 `"hello"` 一樣，本質上就是一個變數名牌，指向記憶體中的可呼叫物件。
當我們寫 `sorted(numbers, key=abs)` 時：
- 我們把 `abs` 這個工具指針傳進去。
- 排序演算法在內部比對元素時，會自動對每個數字呼叫 `abs(x)`。
對於數列 `[-5, 2, -1, 4]`：
- 計算各元素絕對值：`abs(-5)=5`, `abs(2)=2`, `abs(-1)=1`, `abs(4)=4`。
- 比對絕對值順序：$1 < 2 < 4 < 5$。
- 排列原物件：`[-1, 2, 4, -5]`！

#### 3. 初學者常見陷阱：千萬不要加小括號！
請再次銘記：
```python
# 致命錯誤！
sorted(numbers, key=abs())  # 崩潰！TypeError: abs() takes exactly one argument (0 given)
```
加上小括號意味著「你現在就要立刻執行它」，但此時括號內空空如也，當然會引發語法崩潰！我們只需要傳遞**工具名稱 `key=abs`**，呼叫的動作由排序內部自動代勞。

#### 4. APCS 實戰視野
在 APCS 幾何題或誤差逼近題中（例如：找出所有觀測點中離目標基準值最近的點），`key=abs` 經常能幫我們省下數十行繁雜的手動比對迴圈。

In [ ]:
# 範例 12.3.2：內建函數 key=abs 與 key=len 實戰
# 1. 依絕對值排序：找出離 0 度最近的溫度波動
temps = [-15, 3, -2, 8, -1, 10]
print("原始溫度數值：", temps)

# 傳入內建 abs 函數
by_abs = sorted(temps, key=abs)
print("依絕對值大小排序：", by_abs)
print("距離原點 0 最近的溫度：", by_abs[0])
print("距離原點 0 最遠的溫度：", by_abs[-1])

# 2. 依長度排序：單字由短到長
cities = ["Paris", "San Francisco", "Tokyo", "London", "Rome"]
print("\n原始城市清單：", cities)

# 傳入內建 len 函數
by_len = sorted(cities, key=len)
print("依字母長度排序：", by_len)

In [ ]:
# 填空題 12.3.2：利用內建函數快速自訂排序
# 任務 1：將正負溫差數值依照「離 0 度的偏差量（絕對值）」由小到大排序。
# 任務 2：將文章標籤依照「字母長度」由長到短（降序）排序。

deviations = [12, -4, -25, 3, -1, 18]
# 請填入內建絕對值函數名牌
sorted_deviations = sorted(deviations, key=___)
print("依偏差量由小到大：", sorted_deviations)

tags = ["ai", "programming", "algorithm", "python", "it"]
# 請填入內建長度函數名牌與降序參數
sorted_tags = sorted(tags, key=___, reverse=___)
print("標籤依長度降序：", sorted_tags)

In [ ]:
# ==========================================
# [4] Code 練習題 12.3.2
# 任務說明：
# 某量測儀器記錄了 6 筆電壓誤差數值 errors（包含負值）。
# 請使用 key=abs 原地修改方法 .sort() 將誤差由小到大排序，
# 並輸出排序後的串列，以及最小誤差與最大誤差。
#
# 【公開測試資料 1】
# errors = [-0.8, 1.2, -0.1, 0.5, -1.5, 0.2]
# 預期輸出：
# 原地依絕對值排序： [-0.1, 0.2, 0.5, -0.8, 1.2, -1.5]
# 最小絕對誤差值： -0.1
# 最大絕對誤差值： -1.5
#
# 【公開測試資料 2】
# errors = [5, -2, 1]
# 預期輸出：
# 原地依絕對值排序： [1, -2, 5]
# 最小絕對誤差值： 1
# 最大絕對誤差值： 5
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
errors = [-0.8, 1.2, -0.1, 0.5, -1.5, 0.2]
errors.sort(key=abs)
print("原地依絕對值排序：", errors)
print("最小絕對誤差值：", errors[0])
print("最大絕對誤差值：", errors[-1])

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.3.2
# 任務說明：
# 某密碼強度檢核系統有以下一組候選密碼：
# passwords = ["p@ss123", "a", "super_secret_master_key_2026", "admin", "pwd"]
# 請寫出程式碼：
# 1. 使用 sorted 搭配 key=len 找出長度最短的密碼與長度最長的密碼。
# 2. 計算這組密碼的最長與最短長度差距（字元個數差）。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
passwords = ["p@ss123", "a", "super_secret_master_key_2026", "admin", "pwd"]
sorted_pw = sorted(passwords, key=len)
min_pw = sorted_pw[0]
max_pw = sorted_pw[-1]
print(f"最短密碼: '{min_pw}' (長度 {len(min_pw)})")
print(f"最長密碼: '{max_pw}' (長度 {len(max_pw)})")
print(f"長度差距: {len(max_pw) - len(min_pw)} 字元")

### 12.3.3 自訂具名函數作為鍵值：使用 `def get_feature(x):` 傳入 `key=get_feature`

#### 1. 生活故事比喻：特製的珠寶鑑定器
如果海關要檢查的不是一般的長度或重量，而是一批形狀奇形怪狀的古董翡翠。海關需要依照翡翠內部的「含玉比例公式（如：透光度乘上密度）」來排序。
市面上沒有任何現成的尺子能直接量出這個數值。
這時該怎麼辦？海關會聘請一位專屬的珠寶鑑定工程師，打造一台專屬檢驗機：輸入一塊翡翠，機器內部運算自訂公式，最後吐出一個計算結果數值。
在 Python 程式中，這台特製檢驗機就是我們**自訂的具名函數（`def`）**！

#### 2. 底層運作機制：撰寫自訂特徵函數的兩大鐵律
當我們要撰寫一個專門給 `key=` 使用的具名函數時，必須嚴格遵守兩項設計契約：
1. **參數契約（單一輸入）**：  
   函數**必須恰好接收一個參數**（這個參數代表待排序串列中的「單一元素」）。
2. **回傳契約（特徵輸出）**：  
   函數內部計算完成後，**必須透過 `return` 回傳一個可比大小的數值或字串**（如整數、浮點數、文字）。
例如：我們希望依照整數的「個位數字（`x % 10`）」來排序一串數字：
```python
def get_last_digit(num):
    return num % 10

sorted_nums = sorted([38, 12, 95, 41], key=get_last_digit)
# 個位數依序為 1, 2, 5, 8，因此結果為 [41, 12, 95, 38]！
```

#### 3. 初學者常見陷阱：函數內部忘記寫 `return`
許多初學者在自訂 key 函數時，習慣性在函數內部寫 `print(...)` 而忘記寫 `return`：
```python
def bad_key(x):
    last = x % 10
    print(last)  # 致命錯誤！沒有 return，函數預設回傳 None

sorted([38, 12], key=bad_key)  # 崩潰！TypeError: '<' not supported between instances of 'NoneType' and 'NoneType'
```
排序引擎需要的是**具體的比對數值**，而不是在螢幕上印出文字！請務必確認函數有清晰的 `return` 述句。

#### 4. APCS 實戰視野
自訂具名函數結構清晰、易讀性極高。遇到邏輯較繁複的排序條件時（例如字串中母音字母數量、數字的因數個數），寫成具名函數能讓競賽程式碼保持高度的工整與模組化。

In [ ]:
# 範例 12.3.3：自訂具名函數作為排序鍵值
numbers = [142, 85, 31, 79, 64]
print("原始整數清單：", numbers)

# 需求：依照每個數字的「個位數字（最後一位）」由小到大排序
def get_last_digit(n):
    # 個位數公式：n % 10
    return n % 10

# 觀察特徵映射：
for num in numbers:
    print(f"數字 {num} 的個位數特徵是: {get_last_digit(num)}")

# 將自訂函數名牌傳給 key
sorted_by_last_digit = sorted(numbers, key=get_last_digit)
print("\n依個位數字由小到大排序結果：", sorted_by_last_digit)

In [ ]:
# 填空題 12.3.3：依母音字母數量自訂排序
# 任務：計算每個單字中含有幾個英文字母母音 ('aeiou')，並由少到多排序。
words = ["python", "queue", "sky", "beautiful", "tea"]

def count_vowels(word):
    vowels = "aeiou"
    # 計算單字中母音出現的總次數
    cnt = sum(1 for ch in word.lower() if ch in vowels)
    return ___

# 請將 count_vowels 傳入 key
vowel_order = sorted(words, key=___)

print("依母音數量排序單字：", vowel_order)

In [ ]:
# ==========================================
# [4] Code 練習題 12.3.3
# 任務說明：
# 給定一組整數清單 raw_vals。
# 請定義一個具名函數 distance_from_50(x)，計算 x 與數值 50 的距離（即 abs(x - 50)）。
# 將 raw_vals 依「離 50 最近到最遠」排序並印出，
# 且輸出最靠近 50 的數字。
#
# 【公開測試資料 1】
# raw_vals = [42, 65, 53, 20, 48, 80]
# 預期輸出：
# 離 50 距離排序： [48, 53, 42, 65, 20, 80]
# 最接近 50 的數值： 48
#
# 【公開測試資料 2】
# raw_vals = [10, 90, 50]
# 預期輸出：
# 離 50 距離排序： [50, 10, 90]
# 最接近 50 的數值： 50
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def distance_from_50(x):
    return abs(x - 50)

raw_vals = [42, 65, 53, 20, 48, 80]
res = sorted(raw_vals, key=distance_from_50)
print("離 50 距離排序：", res)
print("最接近 50 的數值：", res[0])

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.3.3
# 任務說明：
# 某數學社團在研究「數字各位數之和（Digit Sum）」。
# 給定一組正整數清單：numbers = [123, 45, 90, 11, 204]
# （例如：123 的各位數之和為 1 + 2 + 3 = 6；45 的各位數之和為 4 + 5 = 9）
# 請定義一個具名函數 calc_digit_sum(num)，
# 將 numbers 依照各位數之和「由小到大」排序，並印出排序結果。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def calc_digit_sum(num):
    return sum(int(ch) for ch in str(num))

numbers = [123, 45, 90, 11, 204]
sorted_by_digit_sum = sorted(numbers, key=calc_digit_sum)
print("依各位數之和升序排序：", sorted_by_digit_sum)
for n in sorted_by_digit_sum:
    print(f"  數值 {n:<4} -> 各位數和: {calc_digit_sum(n)}")

### 12.3.4 元組第二欄位排序：`def get_second(item): return item[1]`（成績、年齡排序）

#### 1. 生活故事比喻：學校名冊的欄位對齊
想像一張班級登記表，每一行紀錄包含兩欄：第一欄是「學生姓名」，第二欄是「期末成績」，例如 `("Alice", 95)`、`("Bob", 72)`。
如果直接對這張名冊進行排序，Python 預設會看「第一個欄位（索引 0）」，也就是拿姓名來比 ASCII 字母序（Alice 排在 Bob 前面）。
但期末要發獎學金時，校長要看的是「第二個欄位（索引 1）」——成績！
校長說：「把目光通通對準第二個欄位，誰的分數高、誰的分數低，只看那一欄！」
我們如何教電腦將視線鎖定在特定的欄位上？只需寫一個「提取索引 1 欄位」的專屬小函數！

#### 2. 底層運作機制：多維資料的指定維度降維
當串列中的每個元素是元組（`tuple`）或子串列（`list`）時，例如 `students = [("小明", 88), ("小華", 95), ("小強", 76)]`：
我們定義一個欄位提取函數：
```python
def get_score(student_tuple):
    # student_tuple[0] 是姓名，student_tuple[1] 是成績
    return student_tuple[1]
```
當我們呼叫 `sorted(students, key=get_score)` 時：
- 傳入 `("小明", 88)`，抽取特徵值 `88`。
- 傳入 `("小華", 95)`，抽取特徵值 `95`。
- 傳入 `("小強", 76)`，抽取特徵值 `76`。
排序演算法只比較 `76 < 88 < 95`，從而達成依第二欄位排序的目的。

#### 3. 初學者常見陷阱：索引混淆（Index 0 vs Index 1）
初學同學最容易犯的低級失誤，是把欄位索引搞混：
- `item[0]` 代表第 **一** 個欄位！
- `item[1]` 代表第 **二** 個欄位！
如果不小心寫成 `return item[0]`，排出來的就會是第一欄位的字母序，導致邏輯南轅北轍。

#### 4. APCS 實戰視野
APCS 題目中最常見的二維結構，如平面座標 `(x, y)`、線段起終點 `(start, end)`、商品記錄 `(價格, 庫存)`。能夠熟練地透過具名函數抽取任意欄位進行升序或降序排序，是解開所有二維結構題目的基礎核心技巧。

In [ ]:
# 範例 12.3.4：依元組第二欄位（成績）排序
students = [("Alice", 85), ("Bob", 92), ("Charlie", 78), ("David", 99)]
print("原始學生資料名冊：", students)

# 定義提取第二欄位（成績，索引 1）的函數
def get_score(record):
    return record[1]

# 1. 依成績由低到高升序排列
ranked_asc = sorted(students, key=get_score)
print("\n成績由低到高排序：", ranked_asc)

# 2. 結合 reverse=True，達成成績排行榜（由高到低降序）
ranked_desc = sorted(students, key=get_score, reverse=True)
print("成績榮譽榜（由高到低）：", ranked_desc)
print(f"第一名榜首：{ranked_desc[0][0]}，得分：{ranked_desc[0][1]}")

In [ ]:
# 填空題 12.3.4：依二維座標 Y 軸排序
# 任務：將一組平面座標 (X, Y) 依照 Y 軸座標由小到大排序。
points = [(10, 5), (3, 1), (8, 9), (2, 4)]

def get_y(pt):
    # 請回傳點座標 pt 的 Y 軸（第二個欄位）
    return pt[___]

# 請傳入 get_y 作為 key 排序
sorted_by_y = sorted(points, key=___)

print("依 Y 軸高度由小到大排序：", sorted_by_y)

In [ ]:
# ==========================================
# [4] Code 練習題 12.3.4
# 任務說明：
# 給定一組員工年資紀錄 records，每筆為 (員工代號, 年資年數)。
# 請撰寫程式：
# 1. 定義 get_seniority(rec) 提取年資。
# 2. 將員工清單依照年資由大到小（降序）排序。
# 3. 印出排序後的清單與資歷最深（第一位）的員工代號。
#
# 【公開測試資料 1】
# records = [("E01", 3), ("E02", 12), ("E03", 7), ("E04", 1)]
# 預期輸出：
# 年資由高到低： [('E02', 12), ('E03', 7), ('E01', 3), ('E04', 1)]
# 最資深員工： E02
#
# 【公開測試資料 2】
# records = [("A", 5), ("B", 10)]
# 預期輸出：
# 年資由高到低： [('B', 10), ('A', 5)]
# 最資深員工： B
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def get_seniority(rec):
    return rec[1]

records = [("E01", 3), ("E02", 12), ("E03", 7), ("E04", 1)]
sorted_emp = sorted(records, key=get_seniority, reverse=True)
print("年資由高到低：", sorted_emp)
print("最資深員工：", sorted_emp[0][0])

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.3.4
# 任務說明：
# 某量販店有一批特價水果庫存：
# inventory = [("蘋果", 50, 120), ("香蕉", 25, 300), ("櫻桃", 180, 50), ("芭樂", 35, 200)]
# 每個元組格式為：(水果名稱, 單價, 庫存數量)
# 請定義一個函數 get_total_value(item)，計算每種水果的「庫存總價值（單價 * 庫存數量）」，
# 並將水果清單依庫存總價值由大到小排序後印出。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def get_total_value(item):
    price = item[1]
    qty = item[2]
    return price * qty

inventory = [("蘋果", 50, 120), ("香蕉", 25, 300), ("櫻桃", 180, 50), ("芭樂", 35, 200)]
sorted_inv = sorted(inventory, key=get_total_value, reverse=True)
print("依庫存總資產排序：")
for name, p, q in sorted_inv:
    print(f"  {name:<4}: 單價 {p:3d} x 數量 {q:3d} = 總值 {get_total_value((name, p, q)):6d}")

### 12.3.5 字典走訪排序：以 `sorted(d.items(), key=...)` 依字典的 Value 進行排序

#### 1. 生活故事比喻：投票箱計票排行榜
想像班上舉辦模範生選舉投票，計票員在黑板上記錄了開票結果（以字典存儲）：
`votes = {"Alice": 18, "Bob": 5, "Charlie": 24, "David": 12}`。
如果直接呼叫 `sorted(votes)`，電腦只會把字典的「Key（候選人姓名）」拿出來按字母排好，這根本不是大家關心的結果！
大家關心的是「誰的得票數（Value）最高」！
我們必須請計票員把黑板上的紀錄打包成一張張 `(候選人, 得票數)` 的小紙條（即 `votes.items()`），接著告訴排序官員：「請對準每一張小紙條上的第二格（得票數 Value）進行排序！」

#### 2. 底層運作機制：`d.items()` 的元組化轉換
字典本身是鍵值對映射容器。要對字典的「值（Value）」進行排序，經典標準招式為：
1. **呼叫 `d.items()`**：  
   將字典轉換為鍵值對清單，每個元素都是 `(key, value)` 的雙元組！
2. **傳入提取 Value 的 key 函數**：  
   在 `(key, value)` 結構中，`key` 是第 0 欄，`value` 正好是第 1 欄！
   因此：
   ```python
   def get_val(pair):
       return pair[1]
   sorted_pairs = sorted(d.items(), key=get_val, reverse=True)
   ```
   這樣回傳的就會是依照 Value 由大到小排好的元組串列！

#### 3. 初學者常見陷阱：直接對 `d` 排序卻期待排好 Value
許多初學同學常寫出：
```python
sorted_d = sorted(votes)  # 得到的只是 ['Alice', 'Bob', 'Charlie', 'David']！
```
請牢記：直接對字典物件進行迭代或排序時，**Python 只會拿字典的 Key 來處理！**
若要同時看見 Key 與 Value，**一定要加上 `.items()`**！

#### 4. APCS 實戰視野
在 APCS 統計題型中（例如：輸入一長串文章或數字，統計出現次數最多的前 K 個單字或熱門號碼），「用字典統計次數 ➔ `d.items()` 搭配 Value 排序」是標準得不能再標準的必殺模板題。

In [ ]:
# 範例 12.3.5：依字典的 Value 排序得票數
votes = {"Alice": 18, "Bob": 5, "Charlie": 24, "David": 12}
print("原始字典紀錄：", votes)

# 觀察 votes.items() 的面貌：每一項都是 (姓名, 票數) 的二元組
print("votes.items() 展開：", list(votes.items()))

# 定義提取 Value（票數，索引 1）的函數
def get_vote_count(item_pair):
    return item_pair[1]

# 依得票數由高到低（降序）排序
ranked_results = sorted(votes.items(), key=get_vote_count, reverse=True)

print("\n選舉開票排行榜（由高到低）：")
for rank, (name, count) in enumerate(ranked_results, 1):
    print(f"  第 {rank} 名: {name:<8} 獲得 {count:2d} 票")

In [ ]:
# 填空題 12.3.5：依商品價格將字典項目排序
# 任務：將購物車內的商品品項依照價格由便宜到昂貴（升序）排序。
cart = {"Book": 350, "Pen": 45, "Headphones": 1200, "Eraser": 20}

def get_price(item_pair):
    # 請回傳價格（pair 的第 1 欄）
    return item_pair[___]

# 請將 cart.items() 傳入 sorted，並以 get_price 作為 key
sorted_items = sorted(cart.items(), key=___)

print("購物車品項由便宜到貴：", sorted_items)

In [ ]:
# ==========================================
# [4] Code 練習題 12.3.5
# 任務說明：
# 給定一組單字出現頻率統計字典 freq。
# 請撰寫程式：
# 1. 定義 get_frequency(pair) 提取頻率值。
# 2. 使用 sorted 將 freq.items() 依頻率降序排列。
# 3. 印出排序後的清單，以及出現次數最多的單字與其頻率。
#
# 【公開測試資料 1】
# freq = {"the": 42, "python": 15, "is": 30, "code": 8}
# 預期輸出：
# 詞頻排行： [('the', 42), ('is', 30), ('python', 15), ('code', 8)]
# 最高頻詞彙： the (出現 42 次)
#
# 【公開測試資料 2】
# freq = {"cat": 5, "dog": 10}
# 預期輸出：
# 詞頻排行： [('dog', 10), ('cat', 5)]
# 最高頻詞彙： dog (出現 10 次)
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def get_frequency(pair):
    return pair[1]

freq = {"the": 42, "python": 15, "is": 30, "code": 8}
res = sorted(freq.items(), key=get_frequency, reverse=True)
print("詞頻排行：", res)
print(f"最高頻詞彙： {res[0][0]} (出現 {res[0][1]} 次)")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.3.5
# 任務說明：
# 某線上商店記錄了 5 位客人的消費金額：
# customers = {"UserA": 1500, "UserB": 8200, "UserC": 3400, "UserD": 990, "UserE": 5600}
# 請寫出程式碼：
# 找出消費金額最高的前兩名 VIP 顧客，
# 並計算這兩名 VIP 顧客的消費總合佔全體顧客總營業額的百分比（取小數一位）。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def get_spent(pair):
    return pair[1]

customers = {"UserA": 1500, "UserB": 8200, "UserC": 3400, "UserD": 990, "UserE": 5600}
sorted_customers = sorted(customers.items(), key=get_spent, reverse=True)
top2_sum = sorted_customers[0][1] + sorted_customers[1][1]
total_sum = sum(customers.values())
ratio = (top2_sum / total_sum) * 100
print("前兩大 VIP：", sorted_customers[:2])
print(f"前兩大消費金額合計: {top2_sum} 元，佔全體 {total_sum} 元之 {ratio:.1f}%")

### 12.3.6 排序除錯心法：如何單獨測試 key 函數確認特徵值抽取無誤

#### 1. 生活故事比喻：零件出廠前的獨立品管測試
在火箭發射組裝工廠中，工程師如果把發動機、雷達、導航晶片全部焊接在一起後才按下通電開關，一旦儀表板冒黑煙，沒有人能知道究竟是哪一顆晶片燒毀了。
專業工程師的做法是：發動機做好，單獨放在測試台通電運轉；雷達做好，單獨測試訊號接收。確認每一個獨立零件都精準無誤後，最後才拼裝上火箭。
在編寫自訂 key 函數時也是一樣！
如果你的排序結果亂七八糟，**不要盯著幾百行的排序代碼發呆**！最科學、最高效的除錯心法，就是**「抓出單一樣本資料，單獨餵給 key 函數執行印出」**！

#### 2. 底層運作機制：單元測試（Unit Test）思維在排序上的體現
為什麼具名函數比直接寫匿名 lambda 更有利於初學者除錯？
因為具名函數有名字！你可以隨時在 Python 直譯器或 Colab 中拿單一筆資料呼叫它：
```python
def my_complex_key(item):
    return item[1] * 2 - item[0]

# 除錯心法：先拿一筆樣本測試
sample = (10, 20)
print("測試特徵值輸出：", my_complex_key(sample))
```
如果印出來的結果不是你預期的數值（例如型態錯誤、報出 IndexError、或回傳了 None），你就能立刻在 key 函數內部修改，完全不會影響到整體排序流程。

#### 3. 初學者常見陷阱：在 key 函數裡引發 IndexError 或 KeyError
當串列中的元素長度不一（例如某些字串為空字串 `""`，或元組長度不足）時：
```python
def get_second_char(s):
    return s[1]  # 致命！若字串只有 1 個字或為空字串，立刻爆發 IndexError！

sorted(["hello", "a", "world"], key=get_second_char)
```
排序引擎會對串列裡的「每一個元素」無情呼叫 key 函數。只要有任何一個元素讓 key 函數拋出例外，整個排序操作就會當場夭折中斷。

#### 4. APCS 實戰視野
APCS 實作考題的測資往往包含刁鑽的邊界案例（如長度為 1、數值為 0、極端負數）。在正式將 key 函數交給 `sort` 前，養成「用 1 筆正常測資 + 1 筆極端測資單獨驗證 key 函數」的好習慣，能保證你的排序邏輯百發百中！

In [ ]:
# 範例 12.3.6：單獨測試 key 函數除錯示範
# 題目情境：依照檔案名稱的「副檔名」進行分類排序
# 例如："photo.jpg" 的副檔名是 "jpg"
filenames = ["report.pdf", "data.csv", "summary.docx", "avatar.png", "readme.txt"]

# 撰寫提取副檔名的函數
def get_extension(name):
    # 使用 split(".") 切割，取得最後一段
    parts = name.split(".")
    return parts[-1]

# --- 除錯核心步驟：先單獨測試單一元素 ---
test_sample = "test_image.png"
sample_ext = get_extension(test_sample)
print(f"除錯檢驗：'{test_sample}' 提取到的副檔名是: '{sample_ext}'")
assert sample_ext == "png", "副檔名提取邏輯有誤！"
print("單獨檢驗通過！可以安心交給排序引擎！\n")

# 正式執行排序
sorted_files = sorted(filenames, key=get_extension)
print("依副檔名排序後的檔案清單：", sorted_files)

In [ ]:
# 填空題 12.3.6：修復有瑕疵的 key 函數並單獨測試
# 任務：原本的函數想依據字串第二個字元排序，但遇到單字長度不足 2 時會當機。
# 請加入長度防護，當長度不足 2 時回傳空字串 ""。
words = ["cat", "a", "banana", "dog", ""]

def safe_second_char(s):
    if len(s) < 2:
        return ___
    return s[1]

# 單獨驗證極端測資
print("測試空字串防禦：", repr(safe_second_char("")))
print("測試單字元防禦：", repr(safe_second_char("a")))
print("測試正常單字：", repr(safe_second_char("cat")))

# 驗證通過後安全排序
safe_sorted = sorted(words, key=___)
print("安全排序結果：", safe_sorted)

In [ ]:
# ==========================================
# [4] Code 練習題 12.3.6
# 任務說明：
# 某系統收集了一批日期字串，格式為 "YYYY-MM-DD"（例如 "2026-05-18"）。
# 請撰寫程式：
# 1. 定義 get_month(date_str) 提取其中的「月份（整數）」。
# 2. 先單獨印出 "2026-09-15" 的提取月份，確認為整數 9。
# 3. 使用 key=get_month 將 dates 串列依月份排序並印出。
#
# 【公開測試資料 1】
# dates = ["2026-11-05", "2026-03-21", "2026-08-14", "2026-01-30"]
# 預期輸出：
# 測試單筆月份： 9
# 依月份排序結果： ['2026-01-30', '2026-03-21', '2026-08-14', '2026-11-05']
#
# 【公開測試資料 2】
# dates = ["2025-12-01", "2025-02-28"]
# 預期輸出：
# 測試單筆月份： 9
# 依月份排序結果： ['2025-02-28', '2025-12-01']
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def get_month(date_str):
    return int(date_str.split("-")[1])

print("測試單筆月份：", get_month("2026-09-15"))
dates = ["2026-11-05", "2026-03-21", "2026-08-14", "2026-01-30"]
res = sorted(dates, key=get_month)
print("依月份排序結果：", res)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.3.6
# 任務說明：
# 某賽車遊戲紀錄了選手的 (選手名稱, "分:秒")，例如 ("Mario", "1:25") 代表 1 分 25 秒。
# 請寫出完整程式碼：
# 1. 定義具名函數 time_to_seconds(record)，將第二欄位時間轉換為「總秒數（分*60 + 秒）」。
# 2. 單獨測試單一樣本 ("Luigi", "2:05")，驗證計算結果為 125 秒。
# 3. 給定 racer_data，依花費秒數由少到多（升序，最快者在前）排序並印出冠軍選手名稱。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def time_to_seconds(record):
    m_str, s_str = record[1].split(":")
    return int(m_str) * 60 + int(s_str)

test_racer = ("Luigi", "2:05")
print(f"測試 '{test_racer[0]}' 總秒數驗證: {time_to_seconds(test_racer)} 秒")

racer_data = [("Peach", "1:45"), ("Mario", "1:22"), ("Bowser", "2:10"), ("Toad", "1:30")]
ranked_racers = sorted(racer_data, key=time_to_seconds)
print("賽車排行榜：", ranked_racers)
print(f"最速冠軍：{ranked_racers[0][0]} (耗時 {ranked_racers[0][1]})")

### 學習總結與通關回顧

恭喜你順利通關 **12.3 自訂排序鍵值基礎：key 參數與具名函數（Named Functions）**！

在本單元中，你完成了從「傳統單一數值比較」邁向「任意維度客製化特徵排序」的關鍵思維躍升：
- **`key=` 參數的投射原理（Projection）**：
  - 排序引擎依據 key 函數為每個元素量測出的特徵值比大小。
  - 原物件內部結構完整保留，不被特徵值篡改。
- **內建函數的現成妙用**：
  - `key=len`：依長度大小排序。
  - `key=abs`：依離 0 點距離絕對值大小排序。
  - 語法鐵律：**只傳函數名牌，千萬不可加小括號呼叫 `()`**！
- **自訂具名函數（`def`）客製化提取**：
  - 嚴格遵守「單一輸入參數、單一回傳值」契約。
  - 提取元組第二欄位：`def get_second(item): return item[1]`。
  - 字典 Value 排序必備招式：`sorted(d.items(), key=get_value)`。
- **工程除錯心法**：
  - 單獨餵給 key 函數樣本資料進行獨立驗證，確保無 `IndexError` 與空物件邊界隱患。

---
**下一關預告**：每次排序都要寫一個 3 行的 `def` 函數會不會太繁瑣？能不能一行搞定多個條件的複合排序（例如：成績高者優先，相同時學號小者優先）？下一節 **12.4 匿名函數 lambda 與多準則複合鍵值排序** 將為你解鎖現代 Python 最酷炫、最靈巧的武器庫！